In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Se déplacer directement dans votre dossier de projet existant
%cd "/content/drive/MyDrive/Image Segmentation Project/Project"

# Cloner votre fork GitHub à cet endroit précis
!git clone <BASELINE_REPOSITORY_URL>

%cd <BASELINE_REPO_NAME>
!ls

Mounted at /content/drive
/content/drive/MyDrive/Image Segmentation Project/Project
fatal: destination path '<BASELINE_REPO_NAME>' already exists and is not an empty directory.
<BASELINE_REPO_PATH>
eomt  eval  README.md  step4_qualitative.png  trained_models


In [ ]:
#### STEP 4

# 1. On s'assure d'être dans le bon dossier de code
%cd "<BASELINE_REPO_PATH>"

# 2. On copie les fichiers .bin du dossier des profis vers ton dossier 'trained_models'
!cp "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin" ./trained_models/
!cp "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin" ./trained_models/

# 3. On vérifie que les fichiers sont bien arrivés
!ls trained_models/

<BASELINE_REPO_PATH>
eomt_cityscapes.bin  eomt_finetuned.bin			erfnet_pretrained.pth
eomt_coco.bin	     erfnet_encoder_pretrained.pth.tar


In [ ]:
import sys, os
# Ajoute le dossier eomt au path pour pouvoir importer les modules du repo
sys.path.insert(0, "<BASELINE_REPO_PATH>/eomt")

# On fait les imports nécessaires
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from torchvision.datasets import Cityscapes

# ── 1. PARAMÈTRES ──────────────────────────────────────────────
REPO     = "<BASELINE_REPO_PATH>"
CKPT_CS  = f"{REPO}/trained_models/eomt_cityscapes.bin"
CKPT_CO  = f"{REPO}/trained_models/eomt_coco.bin"
CS_ROOT  = "/content/drive/MyDrive/Image Segmentation Project/Project/2. Cityscapes"  # adapte si différent
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = (1024, 1024)            # taille d'entraînement Cityscapes

# ── 2. CHARGEMENT DES POIDS (identique à _load_ckpt) ──────────
def load_ckpt(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    if "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    ckpt = {k: v for k, v in ckpt.items() if "criterion.empty_weight" not in k}
    return ckpt

ckpt_cs = load_ckpt(CKPT_CS)
ckpt_co = load_ckpt(CKPT_CO)

# Inspecte les clés pour connaître l'architecture encoder
print("=== Cityscapes checkpoint ===")
print("Nb keys :", len(ckpt_cs))
print("Sample keys :", list(ckpt_cs.keys())[:8])
print("\n=== COCO checkpoint ===")
print("Nb keys :", len(ckpt_co))
print("Sample keys :", list(ckpt_co.keys())[:8])

# Cherche num_classes et embed_dim dans les shapes
for k, v in ckpt_cs.items():
    if "class_head" in k:
        print(f"[CS] class_head shape → {v.shape}  (num_classes+1 = {v.shape[0]})")
        break
for k, v in ckpt_co.items():
    if "class_head" in k:
        print(f"[CO] class_head shape → {v.shape}  (num_classes+1 = {v.shape[0]})")
        break
for k, v in ckpt_cs.items():
    if "network.q.weight" in k:
        print(f"[CS] query embed → {v.shape}  (num_q={v.shape[0]}, embed_dim={v.shape[1]})")
        break

=== Cityscapes checkpoint ===
Nb keys : 197
Sample keys : ['network.attn_mask_probs', 'network.encoder.pixel_mean', 'network.encoder.pixel_std', 'network.encoder.backbone.cls_token', 'network.encoder.backbone.reg_token', 'network.encoder.backbone.pos_embed', 'network.encoder.backbone.patch_embed.proj.weight', 'network.encoder.backbone.patch_embed.proj.bias']

=== COCO checkpoint ===
Nb keys : 197
Sample keys : ['network.attn_mask_probs', 'network.encoder.pixel_mean', 'network.encoder.pixel_std', 'network.encoder.backbone.cls_token', 'network.encoder.backbone.reg_token', 'network.encoder.backbone.pos_embed', 'network.encoder.backbone.patch_embed.proj.weight', 'network.encoder.backbone.patch_embed.proj.bias']
[CS] class_head shape → torch.Size([20, 768])  (num_classes+1 = 20)
[CO] class_head shape → torch.Size([134, 768])  (num_classes+1 = 134)
[CS] query embed → torch.Size([100, 768])  (num_q=100, embed_dim=768)


In [ ]:
import timm
from models.eomt import EoMT

def build_eomt(num_classes, ckpt_state_dict, device):

    pos_embed_shape = ckpt_state_dict["network.encoder.backbone.pos_embed"].shape
    num_patches = pos_embed_shape[1]
    img_size = int(num_patches ** 0.5) * 16
    num_q = ckpt_state_dict["network.q.weight"].shape[0]
    print(f"  → img_size={img_size}, num_q={num_q}, num_classes={num_classes}")

    class EncoderWrapper(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = timm.create_model(
                "vit_base_patch16_reg4_gap_256",
                pretrained=False,
                img_size=img_size,
                num_classes=0,
                init_values=1e-5,  # ← active LayerScale avec gamma initialisé
            )
            self.register_buffer("pixel_mean", torch.zeros(1, 3, 1, 1))
            self.register_buffer("pixel_std",  torch.ones(1, 3, 1, 1))

    encoder = EncoderWrapper()
    model = EoMT(
        encoder=encoder,
        num_classes=num_classes,
        num_q=num_q,
        num_blocks=3,
        masked_attn_enabled=True,
    )

    state_dict = {k.replace("network.", ""): v for k, v in ckpt_state_dict.items()}
    incompatible = model.load_state_dict(state_dict, strict=False)

    missing_critical = [k for k in incompatible.missing_keys
                        if not any(x in k for x in ["fc_norm", "cls_token"])]
    unexpected_critical = [k for k in incompatible.unexpected_keys
                           if not any(x in k for x in ["norm", "cls_token"])]

    if missing_critical:
        print("Missing critiques :", missing_critical)
    if unexpected_critical:
        print("Unexpected critiques :", unexpected_critical)
    if not missing_critical and not unexpected_critical:
        print("  ✓ Chargement OK")

    model.eval().to(device)
    return model

model_cs = build_eomt(19,  ckpt_cs, DEVICE)
model_co = build_eomt(133, ckpt_co, DEVICE)
print("\nModèles chargés.")

  → img_size=1024, num_q=100, num_classes=19
  ✓ Chargement OK
  → img_size=640, num_q=200, num_classes=133
  ✓ Chargement OK

Modèles chargés.


In [ ]:
import math
import torch.nn.functional as F
from torchvision.datasets import Cityscapes
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

# ── 4. CLASSES ET COULEURS ─────────────────────────────────────

# Cityscapes : 19 trainIds + palette officielle
CS_CLASSES = [
    "road","sidewalk","building","wall","fence","pole",
    "traffic light","traffic sign","vegetation","terrain","sky",
    "person","rider","car","truck","bus","train","motorcycle","bicycle"
]
CS_COLORS = np.array([
    [128,64,128],[244,35,232],[70,70,70],[102,102,156],[190,153,153],
    [153,153,153],[250,170,30],[220,220,0],[107,142,35],[152,251,152],
    [70,130,180],[220,20,60],[255,0,0],[0,0,142],[0,0,70],[0,60,100],
    [0,80,100],[0,0,230],[119,11,32]
], dtype=np.uint8)

# COCO panoptic : 133 classes (noms abrégés)
COCO_CLASSES = [
    "person","bicycle","car","motorcycle","airplane","bus","train","truck",
    "boat","traffic light","fire hydrant","stop sign","parking meter","bench",
    "bird","cat","dog","horse","sheep","cow","elephant","bear","zebra","giraffe",
    "backpack","umbrella","handbag","tie","suitcase","frisbee","skis","snowboard",
    "sports ball","kite","baseball bat","baseball glove","skateboard","surfboard",
    "tennis racket","bottle","wine glass","cup","fork","knife","spoon","bowl",
    "banana","apple","sandwich","orange","broccoli","carrot","hot dog","pizza",
    "donut","cake","chair","couch","potted plant","bed","dining table","toilet",
    "tv","laptop","mouse","remote","keyboard","cell phone","microwave","oven",
    "toaster","sink","refrigerator","book","clock","vase","scissors","teddy bear",
    "hair drier","toothbrush","banner","blanket","branch","bridge","building-other",
    "bush","cabinet","cage","cardboard","carpet","ceiling-other","ceiling-tile",
    "cloth","clothes","clouds","counter","cupboard","curtain","desk-stuff",
    "dirt","door-stuff","fence","floor-marble","floor-other","floor-stone",
    "floor-tile","floor-wood","flower","fog","food-other","fruit","furniture-other",
    "grass","gravel","ground-other","hill","house","leaves","light","mat","metal",
    "mirror-stuff","moss","mountain","mud","napkin","net","paper","pavement",
    "pillow","plant-other","plastic","platform","playingfield","railing","railroad",
    "river","road","rock","roof","rug","salad","sand","sea","shelf","sky-other",
    "skyscraper","snow","solid-other","stairs","stone","straw","structural-other",
    "table","tent","textile-other","torso-skin","tree","vegetable","wall-brick",
    "wall-concrete","wall-other","wall-panel","wall-stone","wall-tile","wall-wood",
    "water-other","waterdrops","window-blind","window-other","wire-fence","wood",
]
np.random.seed(42)
COCO_COLORS = np.random.randint(0, 255, (133, 3), dtype=np.uint8)

# ── 5. PREPROCESSING ───────────────────────────────────────────

def preprocess_cs(pil_img, device):
    """
    Preprocessing correct pour EoMT-Cityscapes :
    Resize la dimension courte à 1024, garde le ratio (→ 2048×1024)
    puis prend le crop gauche 1024×1024 (zone la plus informative)
    """
    pil_img = pil_img.convert("RGB")
    # Cityscapes est déjà 2048×1024 → pas besoin de resize
    # On prend deux crops : gauche et droite
    arr = np.array(pil_img, dtype=np.uint8)  # (1024, 2048, 3)
    # Crop gauche
    crop = arr[:, :1024, :]  # (1024, 1024, 3)
    t = torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0).to(device)
    return t

def preprocess_co(pil_img, device):
    """Preprocessing pour EoMT-COCO : resize à 640×640"""
    pil_img = pil_img.convert("RGB").resize((640, 640), Image.BILINEAR)
    arr = np.array(pil_img, dtype=np.uint8)
    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)
    return t

# ── 6. INFÉRENCE ───────────────────────────────────────────────

@torch.no_grad()
def predict_semantic(model, pil_img, device):
    """Utilise le vrai pipeline de sliding window du repo"""
    arr = np.array(pil_img.convert("RGB"), dtype=np.uint8)
    img_tensor = torch.from_numpy(arr).permute(2, 0, 1).to(device)  # (3, H, W)

    imgs = [img_tensor]
    img_sizes = [img_tensor.shape[-2:]]

    # Sliding window identique à window_imgs_semantic
    H, W = img_tensor.shape[-2:]
    target = 1024
    scale = max(target / H, target / W)
    new_h, new_w = round(H * scale), round(W * scale)

    pil_resized = pil_img.convert("RGB").resize((new_w, new_h), Image.BILINEAR)
    resized = torch.from_numpy(np.array(pil_resized, dtype=np.uint8)).permute(2, 0, 1).to(device)

    # Crops overlappants
    long_side = max(new_h, new_w)
    num_crops = math.ceil(long_side / target)
    overlap = num_crops * target - long_side
    overlap_per_crop = overlap / (num_crops - 1) if num_crops > 1 else 0

    crops, origins = [], []
    for j in range(num_crops):
        start = int(j * (target - overlap_per_crop))
        end = start + target
        if new_h > new_w:
            crop = resized[:, start:end, :]
        else:
            crop = resized[:, :, start:end]
        crops.append(crop)
        origins.append((0, start, end))

    crops_tensor = torch.stack(crops)  # (N, 3, 1024, 1024)

    mask_logits_per_layer, class_logits_per_layer = model(crops_tensor / 255.0)
    mask_logits  = mask_logits_per_layer[-1]
    class_logits = class_logits_per_layer[-1]
    mask_logits  = F.interpolate(mask_logits, (target, target), mode="bilinear")

    crop_logits = torch.einsum(
        "bqhw, bqc -> bchw",
        mask_logits.sigmoid(),
        class_logits.softmax(-1)[..., :-1],
    )

    # Recompose l'image complète
    logit_sum   = torch.zeros((19, new_h, new_w), device=device)
    logit_count = torch.zeros((19, new_h, new_w), device=device)

    for crop_i, (_, start, end) in enumerate(origins):
        if new_h > new_w:
            logit_sum[:, start:end, :]   += crop_logits[crop_i]
            logit_count[:, start:end, :] += 1
        else:
            logit_sum[:, :, start:end]   += crop_logits[crop_i]
            logit_count[:, :, start:end] += 1

    logits = F.interpolate(
        (logit_sum / logit_count)[None], (H, W), mode="bilinear"
    )[0]

    return logits.argmax(0).cpu()

@torch.no_grad()
def predict_panoptic_rgb(model, img_tensor, stuff_classes=list(range(133))):
    """Retourne image RGB colorisée de la prédiction panoptique"""
    mask_logits_per_layer, class_logits_per_layer = model(img_tensor / 255.0)
    mask_logits  = mask_logits_per_layer[-1]
    class_logits = class_logits_per_layer[-1]
    mask_logits  = F.interpolate(mask_logits, img_tensor.shape[-2:], mode="bilinear")

    scores, classes = class_logits.softmax(-1)[0].max(-1)
    masks = mask_logits[0].sigmoid()
    H, W  = img_tensor.shape[-2:]

    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    keep = classes.ne(class_logits.shape[-1] - 1) & (scores > 0.5)
    if not keep.any():
        return canvas

    mask_ids = (scores[keep][..., None, None] * masks[keep]).argmax(0)
    for k, class_id in enumerate(classes[keep].tolist()):
        final_mask = (mask_ids == k) & (masks[keep][k] >= 0.5)
        if final_mask.sum() == 0:
            continue
        color = COCO_COLORS[class_id % 133]
        canvas[final_mask.cpu().numpy()] = color

    return canvas

# ── 7. CHARGEMENT D'UNE IMAGE CITYSCAPES VAL ──────────────────

cs_dataset = Cityscapes(CS_ROOT, split="val", mode="fine",
                        target_type="semantic")

# Prend 3 images espacées dans le val set
indices = [0, 100, 300]
fig, axes = plt.subplots(len(indices), 4, figsize=(22, 5 * len(indices)))
col_titles = ["Image originale", "GT sémantique", "EoMT-Cityscapes (sém.)", "EoMT-COCO (panopt.)"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11, fontweight="bold")

for row, idx in enumerate(indices):
    pil_img, pil_gt = cs_dataset[idx]
    img_t_cs = preprocess_cs(pil_img, DEVICE)
    img_t_co = preprocess_co(pil_img, DEVICE)

    # GT sémantique → trainIds
    gt_arr = np.array(pil_gt)
    gt_train = np.full_like(gt_arr, 255)
    for cls in Cityscapes.classes:
        if not cls.ignore_in_eval:
            gt_train[gt_arr == cls.id] = cls.train_id

    # Prédictions
    pred_cs     = predict_semantic(model_cs, pil_img, DEVICE).numpy()
    pred_co_rgb = predict_panoptic_rgb(model_co, preprocess_co(pil_img, DEVICE))

    # Colorise pred_cs
    cs_rgb = CS_COLORS[np.clip(pred_cs, 0, 18)]

    # Colorise GT
    gt_rgb = np.zeros((*gt_train.shape, 3), dtype=np.uint8)
    mask_valid = gt_train < 19
    gt_rgb[mask_valid] = CS_COLORS[gt_train[mask_valid]]

    for col, img_show in enumerate([np.array(pil_img), gt_rgb, cs_rgb, pred_co_rgb]):
        axes[row, col].imshow(img_show)
        axes[row, col].axis("off")

# Légende Cityscapes
patches = [mpatches.Patch(color=CS_COLORS[i]/255, label=CS_CLASSES[i]) for i in range(19)]
fig.legend(handles=patches, loc="lower center", ncol=10, fontsize=7,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig("step4_qualitative.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : step4_qualitative.png")

In [ ]:
import torch
import numpy as np

# ── MAPPING FINAL : COCO continu (0-132) → Cityscapes trainID (0-18) ──
# Vérifié sur CLASS_MAPPING de coco_panoptic.py
# 255 = ignore (exclu du calcul mIoU)

COCO_TO_CS = {
    # Véhicules & personnes
    0:   11,  # person        → person
    1:   18,  # bicycle       → bicycle
    2:   13,  # car           → car
    3:   17,  # motorcycle    → motorcycle
    5:   15,  # bus           → bus
    6:   16,  # train         → train
    7:   14,  # truck         → truck

    # Signalisation
    9:    6,  # traffic light → traffic light
    11:   7,  # stop sign     → traffic sign

    # Infrastructure routière
    132:  0,  # road          → road        ✓ corrigé (était 128)
    124:  1,  # pavement      → sidewalk    ✓ corrigé (était 133)

    # Bâti
    80:   2,  # building-other → building
    90:   2,  # house          → building
    109:  3,  # wall-brick     → wall
    110:  3,  # wall-concrete  → wall
    111:  3,  # wall-other     → wall
    114:  3,  # wall-stone     → wall
    117:  3,  # wall-wood      → wall
    82:   4,  # fence          → fence
    130:  4,  # wire-fence     → fence

    # Nature
    81:   8,  # bush          → vegetation
    87:   9,  # sand          → terrain (approximation)
    89:   8,  # grass         → vegetation
    99:   8,  # leaves        → vegetation

    # Ciel
    125: 10,  # sky-other     → sky
}

def remap_coco_to_cityscapes(pred_coco: torch.Tensor) -> torch.Tensor:
    """
    pred_coco : (H, W) LongTensor d'IDs COCO continus (0-132)
    retourne  : (H, W) LongTensor de trainIDs Cityscapes (0-18, 255=ignore)
    """
    pred_cs = torch.full_like(pred_coco, 255)
    for coco_id, cs_id in COCO_TO_CS.items():
        pred_cs[pred_coco == coco_id] = cs_id
    return pred_cs

# ── TEST ──
test = torch.tensor([[0, 2, 132, 50],
                     [1, 7,  80, 124]])
out  = remap_coco_to_cityscapes(test)
print("COCO IDs  :", test)
print("CS trainIDs:", out)
# Attendu : 0→11, 2→13, 132→0, 50→255
#            1→18, 7→14, 80→2, 124→1

COCO IDs  : tensor([[  0,   2, 132,  50],
        [  1,   7,  80, 124]])
CS trainIDs: tensor([[ 11,  13,   0, 255],
        [ 18,  14,   2,   1]])


In [ ]:
# Résultats étape 4 — déjà calculés, pas besoin de relancer
iou_cs_values = [98.2, 86.6, 93.5, 63.2, 59.4, 69.2, 72.3, 77.9, 92.5, 64.5,
                 94.5, 83.7, 67.2, 95.0, 84.6, 88.8, 72.3, 67.4, 79.4]
iou_co_values = [0.0, 0.0, 11.0, 24.9, 1.6, 0.0, 63.6, 31.3, 0.1, 0.0,
                 0.0, 68.7, 0.0, 63.6, 10.2, 58.5, 0.0, 37.7, 68.3]
iou_cs = torch.tensor(iou_cs_values) / 100
iou_co = torch.tensor(iou_co_values) / 100

CS_CLASSES_NAMES = [
    "road","sidewalk","building","wall","fence","pole",
    "traffic light","traffic sign","vegetation","terrain","sky",
    "person","rider","car","truck","bus","train","motorcycle","bicycle"
]

def miou_from_conf(conf):
    tp = conf.diag()
    fp = conf.sum(0) - tp
    fn = conf.sum(1) - tp
    iou = tp.float() / (tp + fp + fn).float().clamp(min=1)
    return iou

print("Résultats étape 4 chargés ✓")

Résultats étape 4 chargés ✓


In [ ]:
import time

CS_CLASSES_NAMES = [
    "road","sidewalk","building","wall","fence","pole",
    "traffic light","traffic sign","vegetation","terrain","sky",
    "person","rider","car","truck","bus","train","motorcycle","bicycle"
]

NUM_CLASSES = 19
IGNORE = 255

# Matrices de confusion : (19, 19) pour chaque modèle
conf_cs = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
conf_co = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)

cs_dataset_eval = Cityscapes(CS_ROOT, split="val", mode="fine", target_type="semantic")
start = time.time()

for idx in range(len(cs_dataset_eval)):
    pil_img, pil_gt = cs_dataset_eval[idx]

    # GT → trainIDs
    gt_arr = np.array(pil_gt, dtype=np.int64)
    gt_train = np.full_like(gt_arr, IGNORE)
    for cls in Cityscapes.classes:
        if not cls.ignore_in_eval:
            gt_train[gt_arr == cls.id] = cls.train_id
    gt_flat = torch.from_numpy(gt_train).flatten()  # (H*W,)

    # ── EoMT-Cityscapes ──
    pred_cs = predict_semantic(model_cs, pil_img, DEVICE).flatten()  # (H*W,)
    mask = gt_flat != IGNORE
    gt_m, pred_m = gt_flat[mask], pred_cs[mask]
    conf_cs += torch.bincount(
        gt_m * NUM_CLASSES + pred_m, minlength=NUM_CLASSES**2
    ).reshape(NUM_CLASSES, NUM_CLASSES)

    # ── EoMT-COCO ──
    img_t_co = preprocess_co(pil_img, DEVICE)
    with torch.no_grad():
        ml, cl = model_co(img_t_co / 255.0)
        ml = F.interpolate(ml[-1], img_t_co.shape[-2:], mode="bilinear")
        logits_co = torch.einsum("bqhw,bqc->bchw", ml.sigmoid(), cl[-1].softmax(-1)[...,:-1])
        pred_co_raw = logits_co[0].argmax(0).cpu()

    pred_co_cs = remap_coco_to_cityscapes(pred_co_raw)
    pred_co_cs_resized = torch.from_numpy(
        np.array(
            Image.fromarray(pred_co_cs.numpy().astype(np.uint8)).resize(
                (gt_train.shape[1], gt_train.shape[0]), Image.NEAREST
            ), dtype=np.int64
        )
    ).flatten()

    mask_co = (gt_flat != IGNORE) & (pred_co_cs_resized != IGNORE)
    gt_m2, pred_m2 = gt_flat[mask_co], pred_co_cs_resized[mask_co]
    conf_co += torch.bincount(
        gt_m2 * NUM_CLASSES + pred_m2, minlength=NUM_CLASSES**2
    ).reshape(NUM_CLASSES, NUM_CLASSES)

    if idx % 50 == 0:
        print(f"[{idx}/{len(cs_dataset_eval)}] {time.time()-start:.0f}s")

# ── Calcul mIoU depuis matrice de confusion ──
def miou_from_conf(conf):
    tp = conf.diag()
    fp = conf.sum(0) - tp
    fn = conf.sum(1) - tp
    iou = tp.float() / (tp + fp + fn).float().clamp(min=1)
    return iou

iou_cs = miou_from_conf(conf_cs)
iou_co = miou_from_conf(conf_co)

print("\n" + "="*60)
print(f"{'Classe':<20} {'EoMT-CS':>10} {'EoMT-COCO':>10}")
print("="*60)
for i, name in enumerate(CS_CLASSES_NAMES):
    print(f"{name:<20} {iou_cs[i]*100:>9.1f}% {iou_co[i]*100:>9.1f}%")
print("="*60)
print(f"{'mIoU':<20} {iou_cs.mean()*100:>9.1f}% {iou_co.mean()*100:>9.1f}%")
print(f"\nTemps total: {time.time()-start:.0f}s")

In [ ]:
#### STEP 5

import timm
from models.eomt import EoMT

def build_eomt_finetune(ckpt_state_dict, num_classes_new, device):
    """
    Construit EoMT-COCO avec une nouvelle class_head pour num_classes_new.
    - On garde tous les poids COCO (backbone, mask_head, queries)
    - On remplace class_head (134 → 20 sorties) par une tête vierge
    - On gèle tout sauf class_head, mask_head et queries
    """
    # Détecte automatiquement img_size et num_q depuis le checkpoint
    pos_embed_shape = ckpt_state_dict["network.encoder.backbone.pos_embed"].shape
    num_patches = pos_embed_shape[1]
    img_size = int(num_patches ** 0.5) * 16  # 640
    num_q = ckpt_state_dict["network.q.weight"].shape[0]  # 200

    class EncoderWrapper(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = timm.create_model(
                "vit_base_patch16_reg4_gap_256",
                pretrained=False,
                img_size=img_size,
                num_classes=0,
                init_values=1e-5,
            )
            self.register_buffer("pixel_mean", torch.zeros(1, 3, 1, 1))
            self.register_buffer("pixel_std",  torch.ones(1, 3, 1, 1))

    encoder = EncoderWrapper()
    model = EoMT(
        encoder=encoder,
        num_classes=num_classes_new,  # 19 classes Cityscapes
        num_q=num_q,
        num_blocks=3,
        masked_attn_enabled=True,
    )

    # Charge les poids COCO — on ignore class_head (shape incompatible)
    state_dict = {k.replace("network.", ""): v for k, v in ckpt_state_dict.items()}
    state_dict = {k: v for k, v in state_dict.items()
                  if "class_head" not in k}  # on retire l'ancienne tête
    incompatible = model.load_state_dict(state_dict, strict=False)

    # Vérifie que seule class_head est manquante (normal)
    missing = [k for k in incompatible.missing_keys
               if "class_head" in k]
    print(f"class_head réinitialisée : {len(missing)} paramètres → OK")

    # Gèle tout le backbone DINOv2
    for name, param in model.named_parameters():
        if "encoder.backbone" in name:
            param.requires_grad = False

    # Compte les paramètres entraînables
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Paramètres entraînables : {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

    return model.to(device)

# Construit le modèle fine-tunable
model_ft = build_eomt_finetune(ckpt_co, num_classes_new=19, device=DEVICE)

class_head réinitialisée : 2 paramètres → OK
Paramètres entraînables : 6,677,780 / 93,574,676 (7.1%)


In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms

# Augmentations pour le train set
# RandomHorizontalFlip : retourne l'image aléatoirement → double la diversité
# ColorJitter : change luminosité/contraste → robustesse aux conditions météo
train_transform_img = transforms.Compose([
    transforms.Resize((640, 640), transforms.InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
])

train_transform_gt = transforms.Compose([
    transforms.Resize((640, 640), transforms.InterpolationMode.NEAREST),
])

cs_train = Cityscapes(
    CS_ROOT,
    split="train",
    mode="fine",
    target_type="semantic",
    transform=train_transform_img,
    target_transform=train_transform_gt,
)

# Collate custom qui convertit le GT PIL → tensor
def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    gts    = torch.stack([
        torch.from_numpy(np.array(item[1], dtype=np.int64))
        for item in batch
    ])
    return images, gts

train_loader = DataLoader(
    cs_train,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn,  # ← gère la conversion PIL→tensor
)

print(f"Train set : {len(cs_train)} images")
print(f"Batches par epoch : {len(train_loader)}")

Train set : 2975 images
Batches par epoch : 1488


In [ ]:
from torch.cuda.amp import GradScaler, autocast

# Loss avec ignore des pixels non annotés
criterion = torch.nn.CrossEntropyLoss(ignore_index=255)

# Optimiseur — seulement sur les paramètres entraînables
optimizer = torch.optim.AdamW(
    [p for p in model_ft.parameters() if p.requires_grad],
    lr=1e-4,        # learning rate standard pour fine-tuning
    weight_decay=0.05
)

# Scaler AMP — gère automatiquement Float16/Float32
scaler = torch.amp.GradScaler('cuda')

print("Loss, optimiseur et AMP configurés ✓")
print(f"Learning rate : 1e-4")
print(f"Paramètres optimisés : {sum(p.numel() for p in model_ft.parameters() if p.requires_grad):,}")

Loss, optimiseur et AMP configurés ✓
Learning rate : 1e-4
Paramètres optimisés : 6,677,780


In [ ]:
import time

# Conversion labelIds → trainIds (même logique que pour le mIoU)
def convert_labels(gt_batch):
    """Convertit un batch de GT labelIds → trainIds Cityscapes"""
    gt_train = torch.full_like(gt_batch, 255, dtype=torch.long)
    for cls in Cityscapes.classes:
        if not cls.ignore_in_eval:
            gt_train[gt_batch == cls.id] = cls.train_id
    return gt_train

# ── BOUCLE D'ENTRAÎNEMENT ──────────────────────────────────────
NUM_EPOCHS = 5
SAVE_PATH  = f"{REPO}/trained_models/eomt_finetuned.bin"

model_ft.train()
best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    start = time.time()

    for batch_idx, (images, gt) in enumerate(train_loader):
        # Convertit les images en uint8 (0-255) pour EoMT
        images = (images * 255).byte().to(DEVICE)  # ToTensor → [0,1], on remet en [0,255]
        gt     = convert_labels(gt.squeeze(1)).to(DEVICE)  # (B, H, W) en trainIds

        optimizer.zero_grad()

        # Forward pass avec AMP (Float16 pour aller plus vite)
        with torch.amp.autocast('cuda'):
            mask_logits_per_layer, class_logits_per_layer = model_ft(images / 255.0)

            # On prend uniquement la dernière couche
            mask_logits  = mask_logits_per_layer[-1]
            class_logits = class_logits_per_layer[-1]

            # Reconstruction des logits par pixel (même einsum qu'avant)
            mask_logits = F.interpolate(mask_logits, images.shape[-2:], mode="bilinear")
            logits = torch.einsum(
                "bqhw, bqc -> bchw",
                mask_logits.sigmoid(),
                class_logits.softmax(-1)[..., :-1],
            )

            loss = criterion(logits, gt)

        # Backward pass avec AMP
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"Batch {batch_idx}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Temps: {time.time()-start:.0f}s")

    avg_loss = epoch_loss / len(train_loader)
    print(f"\n✓ Epoch {epoch+1} terminée | Loss moyenne: {avg_loss:.4f}\n")

    # Sauvegarde le checkpoint à chaque epoch
    torch.save({"state_dict": model_ft.state_dict()}, SAVE_PATH)
    print(f"Checkpoint sauvegardé : {SAVE_PATH}")

print("\nFine-tuning terminé ✓")

Epoch 1/5 | Batch 0/1488 | Loss: 3.0176 | Temps: 9s
Epoch 1/5 | Batch 100/1488 | Loss: 1.9588 | Temps: 185s
Epoch 1/5 | Batch 200/1488 | Loss: 0.5536 | Temps: 340s
Epoch 1/5 | Batch 300/1488 | Loss: 0.8249 | Temps: 494s
Epoch 1/5 | Batch 400/1488 | Loss: 0.3659 | Temps: 644s
Epoch 1/5 | Batch 500/1488 | Loss: 0.2301 | Temps: 797s
Epoch 1/5 | Batch 600/1488 | Loss: 0.2276 | Temps: 954s
Epoch 1/5 | Batch 700/1488 | Loss: 0.5335 | Temps: 1106s
Epoch 1/5 | Batch 800/1488 | Loss: 0.3027 | Temps: 1262s
Epoch 1/5 | Batch 900/1488 | Loss: 0.4846 | Temps: 1413s
Epoch 1/5 | Batch 1000/1488 | Loss: 0.2943 | Temps: 1565s
Epoch 1/5 | Batch 1100/1488 | Loss: 0.2712 | Temps: 1717s
Epoch 1/5 | Batch 1200/1488 | Loss: 0.4318 | Temps: 1868s
Epoch 1/5 | Batch 1300/1488 | Loss: 0.2618 | Temps: 2018s
Epoch 1/5 | Batch 1400/1488 | Loss: 0.4089 | Temps: 2176s

✓ Epoch 1 terminée | Loss moyenne: 0.6078

Checkpoint sauvegardé : <BASELINE_REPO_PATH>/trained_models/eomt_finetuned.bin
Epoch 2/5 | Batch 0/1488 | L

In [ ]:
model_ft = build_eomt_finetune(ckpt_co, num_classes_new=19, device=DEVICE)
ft_ckpt = torch.load(f"{REPO}/trained_models/eomt_finetuned.bin", map_location="cpu", weights_only=True)
model_ft.load_state_dict(ft_ckpt["state_dict"])
model_ft.eval()
print("Modèle fine-tuné rechargé ✓")

class_head réinitialisée : 2 paramètres → OK
Paramètres entraînables : 6,677,780 / 93,574,676 (7.1%)
Modèle fine-tuné rechargé ✓


In [ ]:
import time

def preprocess_co(pil_img, device):
    """Preprocessing pour EoMT-COCO/fine-tuné : resize à 640×640"""
    pil_img = pil_img.convert("RGB").resize((640, 640), Image.BILINEAR)
    arr = np.array(pil_img, dtype=np.uint8)
    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)
    return t


# Charge le modèle fine-tuné
model_ft.eval()

# Matrice de confusion pour le modèle fine-tuné
conf_ft = torch.zeros(19, 19, dtype=torch.long)

cs_dataset_eval = Cityscapes(CS_ROOT, split="val", mode="fine", target_type="semantic")
start = time.time()

for idx in range(len(cs_dataset_eval)):
    pil_img, pil_gt = cs_dataset_eval[idx]

    # GT → trainIDs
    gt_arr = np.array(pil_gt, dtype=np.int64)
    gt_train = np.full_like(gt_arr, 255)
    for cls in Cityscapes.classes:
        if not cls.ignore_in_eval:
            gt_train[gt_arr == cls.id] = cls.train_id
    gt_flat = torch.from_numpy(gt_train).flatten()

    # Prédiction fine-tuné — même preprocessing que COCO (640×640)
    img_co = preprocess_co(pil_img, DEVICE)
    with torch.no_grad():
        ml, cl = model_ft(img_co / 255.0)
        ml = F.interpolate(ml[-1], img_co.shape[-2:], mode="bilinear")
        logits = torch.einsum("bqhw,bqc->bchw", ml.sigmoid(), cl[-1].softmax(-1)[...,:-1])
        pred = logits[0].argmax(0).cpu()

    # Resize vers taille GT
    pred_resized = torch.from_numpy(
        np.array(
            Image.fromarray(pred.numpy().astype(np.uint8)).resize(
                (gt_train.shape[1], gt_train.shape[0]), Image.NEAREST
            ), dtype=np.int64
        )
    ).flatten()

    mask = gt_flat != 255
    gt_m, pred_m = gt_flat[mask], pred_resized[mask]
    conf_ft += torch.bincount(
        gt_m * 19 + pred_m, minlength=19**2
    ).reshape(19, 19)

    if idx % 50 == 0:
        print(f"[{idx}/{len(cs_dataset_eval)}] {time.time()-start:.0f}s")

# Résultats — comparaison des 3 modèles
iou_ft = miou_from_conf(conf_ft)

print("\n" + "="*70)
print(f"{'Classe':<20} {'EoMT-CS':>10} {'EoMT-COCO':>10} {'Fine-tuné':>10}")
print("="*70)
for i, name in enumerate(CS_CLASSES_NAMES):
    print(f"{name:<20} {iou_cs[i]*100:>9.1f}% {iou_co[i]*100:>9.1f}% {iou_ft[i]*100:>9.1f}%")
print("="*70)
print(f"{'mIoU':<20} {iou_cs.mean()*100:>9.1f}% {iou_co.mean()*100:>9.1f}% {iou_ft.mean()*100:>9.1f}%")

[0/500] 2s
[50/500] 49s
[100/500] 88s
[150/500] 127s
[200/500] 167s
[250/500] 208s
[300/500] 266s
[350/500] 327s
